In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import numpy as np
import cv2
from ball_metrics import CheckBall  # Import the ball detection function

# Load Dataset

In [3]:
dataset = load_dataset("mspitzna/physicsgen",name='ball_roll',trust_remote_code=True)

Generating train split: 27840 examples [00:11, 2430.04 examples/s]
Generating test split: 1800 examples [00:00, 4626.52 examples/s]
Generating validation split: 50 examples [00:00, 2798.48 examples/s]


In [5]:
train = dataset['train']
test = dataset['test']
val = dataset['validation']

In [6]:
sample = train[0]
print(sample.keys())


dict_keys(['ImgName', 'StartHeight', 'GroundIncli', 'InputTime', 'TargetTime', 'input_image', 'target_image'])


In [ ]:


class BallRollDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        sample = self.samples[idx]
        # Images: (H, W, C) --> (C, H, W)
        input_img = torch.tensor(sample["input_image"], dtype=torch.float32).permute(2, 0, 1) / 255.
        target_img = torch.tensor(sample["target_image"], dtype=torch.float32).permute(2, 0, 1) / 255.
        # Times
        input_time = torch.tensor(float(sample["InputTime"]), dtype=torch.float32)
        target_time = torch.tensor(float(sample["TargetTime"]), dtype=torch.float32)
        return input_img, target_img, input_time, target_time




In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(64 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(64 * 16 * 16, latent_dim)
    def forward(self, x):
        h = self.conv(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64 * 16 * 16)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, 64, 16, 16)
        return self.deconv(h)

In [ ]:
# ==== Neural ODE block ====
class LatentODEFunc(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 128), nn.Tanh(),
            nn.Linear(128, latent_dim)
        )
    def forward(self, t, z):
        # t: scalar, z: (batch, latent_dim)
        batch_size = z.shape[0]
        t_vec = t.expand(batch_size, 1)
        inp = torch.cat([z, t_vec], dim=1)
        return self.net(inp)

In [ ]:
class ODEVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.odefunc = LatentODEFunc(latent_dim)
        self.decoder = Decoder(latent_dim)
    def forward(self, x, t0, t1):
        mu, logvar = self.encoder(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z0 = mu + eps * std
        times = torch.stack([t0, t1], dim=0)  # (2, batch)
        # Assume times are scalars, so we need to (batch, two_times)
        # Use odeint for each sample in batch
        zT_list = []
        for i in range(z0.shape[0]):
            curr_times = torch.tensor([t0[i].item(), t1[i].item()], device=x.device)
            zT = odeint(self.odefunc, z0[i:i+1], curr_times)[-1].squeeze(0)
            zT_list.append(zT)
        zT = torch.stack(zT_list, dim=0)
        out = self.decoder(zT)
        return out, mu, logvar

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    recon = F.mse_loss(recon_x, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kld


In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
latent_dim = 32
model = ODEVAE(latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

In [ ]:
train_ds = BallRollDataset(dataset["train"])[:1000]
val_ds   = BallRollDataset(dataset["validation"])
test_ds  = BallRollDataset(dataset["test"])
train_loader = DataLoader(train_ds, batch_size=16,num_workers=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=16, num_workers=2)

In [ ]:
def compute_position_metrics(pred_img, target_img, image_size=64):
    # Convert from pytorch tensor (C,H,W) to opencv format (H,W,C)
    pred_np = (pred_img.detach().cpu().permute(1,2,0).numpy() * 255).astype(np.uint8)
    target_np = (target_img.detach().cpu().permute(1,2,0).numpy() * 255).astype(np.uint8)
    
    # Get ball positions from both images
    _, pred_x, pred_y, _, _, _, pred_center_ok, _, _, _ = CheckBall(pred_np, image_size, DEBUG=False)
    _, true_x, true_y, _, _, _, true_center_ok, _, _, _ = CheckBall(target_np, image_size, DEBUG=False)
    
    if pred_center_ok and true_center_ok:
        center_x_err = true_x - pred_x
        center_y_err = true_y - pred_y
    else:
        center_x_err = float('inf')
        center_y_err = float('inf')
        
    return center_x_err, center_y_err

# Training loop with metrics
num_epochs = 10
image_size = 64  # Make sure this matches your actual image size

avg_losses = []
avg_x_errors = []
avg_y_errors = []

for epoch in range(num_epochs):
    epoch_losses = []
    epoch_x_errors = []
    epoch_y_errors = []
    
    for input_img, target_img, input_time, target_time in train_loader:
        input_img = input_img.to(device)
        target_img = target_img.to(device)
        input_time = input_time.to(device)
        target_time = target_time.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        y_pred, mu, logvar = model(input_img, input_time, target_time)
        
        # Compute VAE loss
        loss = vae_loss(y_pred, target_img, mu, logvar)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Compute position errors for each image in batch
        batch_x_errors = []
        batch_y_errors = []
        
        for i in range(y_pred.shape[0]):
            x_err, y_err = compute_position_metrics(y_pred[i], target_img[i], image_size)
            if x_err != float('inf'):
                batch_x_errors.append(abs(x_err))
                batch_y_errors.append(abs(y_err))
        
        # Store metrics
        epoch_losses.append(loss.item())
        if batch_x_errors:  # Only if valid errors were found
            epoch_x_errors.extend(batch_x_errors)
            epoch_y_errors.extend(batch_y_errors)
    
    # Compute epoch averages
    avg_loss = np.mean(epoch_losses)
    avg_x_error = np.mean(epoch_x_errors) if epoch_x_errors else float('inf')
    avg_y_error = np.mean(epoch_y_errors) if epoch_y_errors else float('inf')
    
    # Store for plotting
    avg_losses.append(avg_loss)
    avg_x_errors.append(avg_x_error)
    avg_y_errors.append(avg_y_error)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Avg X Error: {avg_x_error:.2f} pixels")
    print(f"Avg Y Error: {avg_y_error:.2f} pixels")
    print("-" * 50)



In [ ]:
print("Training complete.")

# Plot training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(avg_losses)
plt.title('VAE Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 2)
plt.plot(avg_x_errors)
plt.title('X Position Error')
plt.xlabel('Epoch')
plt.ylabel('Pixels')

plt.subplot(1, 3, 3)
plt.plot(avg_y_errors)
plt.title('Y Position Error')
plt.xlabel('Epoch')
plt.ylabel('Pixels')

plt.tight_layout()
plt.show()